# PACE: Predictive Allocation for Club Excellence
## Multi-Objective Optimization Analysis

**Student Name:** Justin Null
**Target Company/Industry:** Life Time Group Holdings (NYSE: LTH), premium health and fitness club operations
**Problem:** How should a Life Time center split its weekly flexible labor (floor staff, group fitness instructors, trainers on shift, childcare) between peak and off-peak hours?
**Objectives:** Minimize labor cost per member visit vs. maximize the Member Experience Score

---

### Project Overview

Life Time sells an experience, not a gym pass. Its center operations line, its largest cost, ran 51.6% of revenue in Q1 2026, and management tied part of the increase to busier clubs. Staffing every hour for the early-morning and evening rush protects the member experience that justifies premium dues ($930 average revenue per center membership in Q1 2026), but those same people sit idle at midday and inflate cost per visit. Trimming labor to the "average" day saves money but creates sold-out classes, unattended floors and childcare waitlists at exactly the moments members care about.

This notebook is the **optimization engine** for PACE. It simulates a representative center, evaluates **100 weekly staffing plans**, and maps the full Pareto frontier between "lean" and "luxurious". I control two **decision variables**:

| Decision variable | Range | Meaning |
|---|---|---|
| `peak_coverage_pct` | 60–150% | Staffing in peak windows (weekday 5–9 AM and 4–8 PM, weekend 8 AM–12 PM) as a % of the demand-based target |
| `offpeak_coverage_pct` | 40–130% | Staffing in all other open hours as a % of the same target |

Phase 2 will add the constraint-aware decision agent on top of this frontier.

In [1]:
# --- Setup: imports, plotting style, project paths -------------------------
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
IMG_DIR = PROJECT_ROOT / "images"
IMG_DIR.mkdir(exist_ok=True)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

# Chart palette: colorblind-validated categorical hues and neutral ink
COLORS = {
    "frontier":  "#2a78d6",   # Pareto-optimal plans
    "strategy":  "#eb6834",   # highlighted strategic points
    "dominated": "#a3a29c",   # inefficient plans (recessive gray)
    "cost":      "#1baf7a",   # cost-efficiency series
    "experience":"#6250d6",   # member-experience series
    "ink":       "#0b0b0b",
    "ink_2":     "#52514e",
    "grid":      "#e4e3df",
}
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": COLORS["ink_2"], "axes.labelcolor": COLORS["ink"],
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": COLORS["grid"], "grid.linewidth": 0.8,
    "xtick.color": COLORS["ink_2"], "ytick.color": COLORS["ink_2"],
    "font.size": 10.5, "axes.titlesize": 13, "axes.titleweight": "bold",
    "legend.frameon": False,
})
print("Environment ready | pandas", pd.__version__, "| numpy", np.__version__)

Environment ready | pandas 3.0.5 | numpy 2.5.3


## Section 1: Data Acquisition

**Strategy: synthetic data (LLM + Python), as approved in the proposal.** Individual clubs don't publish hourly traffic or shift data, so I prompted an LLM (Claude) to write a **seeded simulator** to my spec rather than paste a CSV by hand. A hand-typed CSV can't keep cost, hours and service levels arithmetically consistent, and it can't be reproduced. The exact prompt is in [`data/data_generation_prompt.txt`](data/data_generation_prompt.txt), and the calibration sources and validation notes are in [`data/DATA_SOURCE.md`](data/DATA_SOURCE.md).

How the simulator works ([`src/generate_data.py`](src/generate_data.py)):

1. **Demand.** It simulates 8 weeks × 7 days × 17 open hours of check-ins for one center: ≈4,410 memberships (837,903 center memberships ÷ 190 centers, Q1 2026) × 3.0 visits per week. Weekdays get a morning and an evening rush plus a lunch bump, and weekends get a late-morning peak. Week-, day- and hour-level noise means no two days repeat.
2. **Plans.** It scores every combination of the two coverage levels (10 × 10 = 100 plans) on the **same** 8 weeks (common random numbers), so differences between plans come from the decision, not from luck.
3. **Physics per hour.**
   - Floor help requests are an **M/M/c (Erlang C) queue**, which gives time-to-assistance.
   - Classes are capped at 3 studios × 35 spots, which gives fill rate and waitlist.
   - Childcare is capped at a 1:6 ratio with a two-adult minimum, which gives turnaways. A turned-away parent usually skips the visit.
4. **Cost.** Floor and childcare hours, plus pay per class, with 1.25× burden and 1.5× overtime past roster hours.

In [2]:
# Reproducibility check: regenerate the dataset from the seeded simulator and
# confirm it is identical to the committed CSV.
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from generate_data import build_dataset, PARAMS

regenerated, _ = build_dataset()
saved = pd.read_csv(DATA_DIR / "staffing_scenarios.csv")
pd.testing.assert_frame_equal(saved, regenerated, check_dtype=False, atol=1e-4)
print(f"Reproducibility check passed: simulator (seed={PARAMS['seed']}) reproduces "
      f"staffing_scenarios.csv exactly ({len(saved)} rows).")

Reproducibility check passed: simulator (seed=2026) reproduces staffing_scenarios.csv exactly (100 rows).


In [3]:
# Load the analysis dataset (one row per weekly staffing plan)
df = pd.read_csv(DATA_DIR / "staffing_scenarios.csv")
demand = pd.read_csv(DATA_DIR / "hourly_demand.csv")

# Names used throughout the notebook
DV1, DV2 = "peak_coverage_pct", "offpeak_coverage_pct"
OBJECTIVE_1 = "labor_cost_per_visit"       # MINIMIZE
OBJECTIVE_2 = "member_experience_score"    # MAXIMIZE
LABELS = {
    DV1: "Peak-hour staffing coverage (%)",
    DV2: "Off-peak staffing coverage (%)",
    OBJECTIVE_1: "Labor cost per member visit ($)",
    OBJECTIVE_2: "Member Experience Score (0-100)",
}

print("Data loaded successfully!")
print(f"Shape: {df.shape}   (simulated demand: {demand.shape[0]} center-hours)")
print("\nFirst 5 rows:")
df.head()

Data loaded successfully!
Shape: (100, 22)   (simulated demand: 952 center-hours)

First 5 rows:


,plan_id,peak_coverage_pct,offpeak_coverage_pct,labor_cost_per_visit,member_experience_score,weekly_labor_cost,weekly_visits,floor_staff_hours,childcare_staff_hours,classes_per_week,total_labor_hours,overtime_hours,assist_score,class_fit_score,childcare_score,avg_time_to_assist_min,peak_time_to_assist_min,avg_class_fill_rate,class_waitlist_per_week,childcare_turnaways_per_week,cost_per_visit_week_sd,mes_week_sd
0,P001,60,40,0.8619,45.7929,11378.7891,13202.135,262.0000,154.0625,41.000,457.0625,0.0,19.9769,43.5473,90.2424,12.2250,15.0963,0.9636,1531.8378,57.5789,0.0380,1.8323
1,P002,60,50,0.8685,46.3597,11465.9766,13202.135,262.1250,154.0625,42.500,458.6875,0.0,19.9769,45.1667,90.2424,12.2250,15.0963,0.9624,1483.0241,57.5789,0.0358,1.6936
2,P003,60,60,0.8834,47.4444,11662.8516,13202.135,262.4375,154.0625,45.875,462.3750,0.0,19.9769,48.2657,90.2424,12.2250,15.0963,0.9487,1391.4211,57.5789,0.0337,1.4878
3,P004,60,70,0.9079,48.6740,11986.2891,13202.135,263.0625,154.0625,51.375,468.5000,0.0,19.9769,51.7789,90.2424,12.2225,15.0963,0.9172,1265.3416,57.5789,0.0322,1.4837
4,P005,60,80,0.9251,49.3649,12212.6953,13202.135,263.7500,154.0625,55.125,472.9375,0.0,20.2109,53.4855,90.2424,12.1086,15.0963,0.8941,1189.5647,57.5789,0.0301,1.3848


## Section 2: Data Exploration

## Section 3: Pareto Frontier Calculation

## Section 4: Strategic Analysis

## Section 5: Visualization